In [1]:
#!/usr/bin/env python3
"""
ingest_prices.py

Descarga SECUENCIAL del histórico diario de precios OHLCV
para las ~3.000 empresas del universo multifactor.

Patrón incremental:
  - Primera corrida: baja 2 años de historia
  - Corridas siguientes: solo el delta desde el último día en DB
  - Cero duplicados: ON CONFLICT DO NOTHING

Fuente: FMP /stable/historical-price-eod/dividend-adjusted
"""

import os
import time
import logging
import requests
import subprocess
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from dotenv import load_dotenv
import psycopg2
import psycopg2.extras

# ── ENV ────────────────────────────────────────────────────────────────────────
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
API_KEY           = os.getenv("FMP_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

BASE_URL  = "https://financialmodelingprep.com/stable/historical-price-eod/dividend-adjusted"
DIAS_HIST = 730    # historia inicial: 2 años
SLEEP_OK  = 0.25   # segundos entre requests exitosas
SLEEP_ERR = 5.0    # segundos tras error
RUN_ID    = datetime.now().strftime("%Y%m%d_%H%M")

# ── Logging ────────────────────────────────────────────────────────────────────
LOG_DIR  = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/ingest_prices_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logging.info(f"=== START ingest_prices | run_id: {RUN_ID} ===")

# ── Keep Mac awake ─────────────────────────────────────────────────────────────
try:
    caffeinate = subprocess.Popen(["caffeinate"])
except Exception:
    caffeinate = None

# ── DB ─────────────────────────────────────────────────────────────────────────
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD, host=POSTGRES_HOST, port=POSTGRES_PORT
    )

def log_db(conn, ticker, status, message):
    try:
        with conn.cursor() as cur:
            cur.execute("""
                INSERT INTO infraestructura.update_logs
                (schema_name, table_name, ticker, status, message)
                VALUES (%s, %s, %s, %s, %s)
            """, ('api_raw', 'multifactor_prices', ticker, status, message))
            conn.commit()
    except Exception:
        conn.rollback()

def obtener_tickers(conn) -> list[str]:
    with conn.cursor() as cur:
        cur.execute("SELECT ticker FROM universos.stock_opciones_2026")
        return [r[0] for r in cur.fetchall()]

def ultima_fecha_en_db(conn, ticker: str):
    """Devuelve la última fecha disponible en DB para este ticker. None si no hay datos."""
    with conn.cursor() as cur:
        cur.execute(
            "SELECT MAX(fecha) FROM api_raw.multifactor_prices WHERE ticker = %s",
            (ticker,)
        )
        return cur.fetchone()[0]

# ── Descarga FMP ───────────────────────────────────────────────────────────────
def descargar_precios(ticker: str, desde: str, hasta: str) -> pd.DataFrame | None:
    """
    Descarga OHLCV ajustado desde FMP.
    Devuelve DataFrame limpio o None si falla.
    """
    params = {
        "symbol": ticker,
        "apikey": API_KEY,
        "from":   desde,
        "to":     hasta,
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=30)

        if r.status_code == 429:
            logging.warning(f"  ⏳ Rate limit en {ticker} — esperando 60s")
            time.sleep(60)
            r = requests.get(BASE_URL, params=params, timeout=30)

        if r.status_code != 200:
            return None

        data = r.json()
        if not data:
            return None

        df = pd.DataFrame(data)
        df["date"] = pd.to_datetime(df["date"])

        # Renombrar columnas al esquema de la tabla
        rename = {
            "date":     "fecha",
            "open":     "open_adj",
            "high":     "high_adj",
            "low":      "low_adj",
            "adjClose": "close_adj",
            "volume":   "volume",
        }
        df = df.rename(columns=rename)

        cols = ["fecha", "open_adj", "high_adj", "low_adj", "close_adj", "volume"]
        cols_presentes = [c for c in cols if c in df.columns]
        df = df[cols_presentes].copy()

        # Limpiar tipos
        df["fecha"] = df["fecha"].dt.date
        for col in ["open_adj", "high_adj", "low_adj", "close_adj"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        if "volume" in df.columns:
            df["volume"] = pd.to_numeric(df["volume"], errors="coerce").astype("Int64")

        df = df.dropna(subset=["close_adj"])
        return df if not df.empty else None

    except Exception as e:
        logging.warning(f"  ✗ Error descargando {ticker}: {e}")
        return None

# ── INSERT en batch ────────────────────────────────────────────────────────────
def insertar_precios(conn, ticker: str, df: pd.DataFrame) -> int:
    """
    Inserta filas en batch. ON CONFLICT DO NOTHING — nunca duplica.
    Devuelve cantidad de filas procesadas.
    """
    if df is None or df.empty:
        return 0

    sql = """
        INSERT INTO api_raw.multifactor_prices
            (ticker, fecha, open_adj, high_adj, low_adj, close_adj, volume, run_id)
        VALUES
            (%(ticker)s, %(fecha)s, %(open_adj)s, %(high_adj)s,
             %(low_adj)s, %(close_adj)s, %(volume)s, %(run_id)s)
        ON CONFLICT (ticker, fecha) DO NOTHING
    """

    filas = []
    for _, row in df.iterrows():
        filas.append({
            "ticker":   ticker,
            "fecha":    row["fecha"],
            "open_adj": float(row["open_adj"]) if "open_adj" in row and pd.notna(row["open_adj"]) else None,
            "high_adj": float(row["high_adj"]) if "high_adj" in row and pd.notna(row["high_adj"]) else None,
            "low_adj":  float(row["low_adj"])  if "low_adj"  in row and pd.notna(row["low_adj"])  else None,
            "close_adj":float(row["close_adj"]),
            "volume":   int(row["volume"])      if "volume"   in row and pd.notna(row["volume"])   else None,
            "run_id":   RUN_ID,
        })

    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, sql, filas, page_size=500)
    conn.commit()
    return len(filas)

# ── MAIN ───────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print(f"  INGEST PRICES — {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  run: {RUN_ID}")
    print(f"{'='*65}\n")

    conn    = get_conn()
    tickers = obtener_tickers(conn)
    total   = len(tickers)
    hoy     = date.today()

    logging.info(f"Tickers a procesar: {total}")

    ok, fail, skip = 0, 0, 0

    for i, ticker in enumerate(tickers, 1):

        ultima = ultima_fecha_en_db(conn, ticker)

        if ultima is None:
            # Primera vez — bajar histórico completo
            desde = (hoy - timedelta(days=DIAS_HIST)).strftime("%Y-%m-%d")
            modo  = "histórico"
        else:
            # Incremental — solo desde el día siguiente al último
            desde = (ultima + timedelta(days=1)).strftime("%Y-%m-%d")
            modo  = "incremental"

            if desde > hoy.strftime("%Y-%m-%d"):
                skip += 1
                continue   # ya está al día

        hasta = hoy.strftime("%Y-%m-%d")

        try:
            df = descargar_precios(ticker, desde, hasta)

            if df is None:
                log_db(conn, ticker, "NO_DATA", f"Sin datos desde {desde}")
                fail += 1
                time.sleep(SLEEP_ERR)
                continue

            n = insertar_precios(conn, ticker, df)
            log_db(conn, ticker, "success", f"{modo} | {n} filas")
            ok += 1

            if i % 100 == 0 or modo == "histórico":
                logging.info(f"  [{i}/{total}] {ticker:6} ({modo}) → {n} filas")

        except Exception as e:
            conn.rollback()
            log_db(conn, ticker, "exception", str(e))
            fail += 1
            logging.warning(f"  ✗ {ticker}: {e}")
            time.sleep(SLEEP_ERR)

        time.sleep(SLEEP_OK)

    conn.close()

    print(f"\n{'─'*65}")
    print(f"  OK: {ok} | FAIL: {fail} | SKIP (ya al día): {skip}")
    print(f"  Total procesados: {total}")
    print(f"\n{'='*65}")
    print(f"  Pipeline completo — {datetime.now().strftime('%H:%M:%S')}")
    print(f"{'='*65}\n")

    logging.info(f"FIN | OK={ok} | FAIL={fail} | SKIP={skip} | TOTAL={total}")

# ──────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        main()
    finally:
        if caffeinate:
            caffeinate.terminate()

2026-04-08 11:01:22,953 | INFO | === START ingest_prices | run_id: 20260408_1101 ===
2026-04-08 11:01:23,097 | INFO | Tickers a procesar: 3027



  INGEST PRICES — 2026-04-08 11:01  |  run: 20260408_1101



2026-04-08 11:01:23,961 | INFO |   [1/3027] BBY    (histórico) → 502 filas
2026-04-08 11:01:25,003 | INFO |   [2/3027] AIR    (histórico) → 502 filas
2026-04-08 11:01:26,044 | INFO |   [3/3027] ABT    (histórico) → 502 filas
2026-04-08 11:01:27,264 | INFO |   [4/3027] ACU    (histórico) → 502 filas
2026-04-08 11:01:28,359 | INFO |   [5/3027] BKTI   (histórico) → 502 filas
2026-04-08 11:01:29,418 | INFO |   [6/3027] AMD    (histórico) → 502 filas
2026-04-08 11:01:30,467 | INFO |   [7/3027] APD    (histórico) → 502 filas
2026-04-08 11:01:31,520 | INFO |   [8/3027] CECO   (histórico) → 502 filas
2026-04-08 11:01:32,574 | INFO |   [9/3027] MATX   (histórico) → 502 filas
2026-04-08 11:01:33,608 | INFO |   [10/3027] ALX    (histórico) → 502 filas
2026-04-08 11:01:34,646 | INFO |   [11/3027] ALCO   (histórico) → 502 filas
2026-04-08 11:01:35,716 | INFO |   [12/3027] LNG    (histórico) → 502 filas
2026-04-08 11:01:36,762 | INFO |   [13/3027] SWKS   (histórico) → 502 filas
2026-04-08 11:01:37,8

2026-04-08 11:03:17,711 | INFO |   [109/3027] ESCA   (histórico) → 502 filas
2026-04-08 11:03:18,749 | INFO |   [110/3027] ESP    (histórico) → 502 filas
2026-04-08 11:03:19,816 | INFO |   [111/3027] KINS   (histórico) → 502 filas
2026-04-08 11:03:20,858 | INFO |   [112/3027] BOOM   (histórico) → 502 filas
2026-04-08 11:03:21,900 | INFO |   [113/3027] XOM    (histórico) → 502 filas
2026-04-08 11:03:22,938 | INFO |   [114/3027] SRCE   (histórico) → 502 filas
2026-04-08 11:03:23,980 | INFO |   [115/3027] FRT    (histórico) → 502 filas
2026-04-08 11:03:25,039 | INFO |   [116/3027] FITBI  (histórico) → 502 filas
2026-04-08 11:03:26,110 | INFO |   [117/3027] FFIN   (histórico) → 502 filas
2026-04-08 11:03:27,166 | INFO |   [118/3027] USB    (histórico) → 502 filas
2026-04-08 11:03:28,214 | INFO |   [119/3027] TRMK   (histórico) → 502 filas
2026-04-08 11:03:29,256 | INFO |   [120/3027] MTB    (histórico) → 502 filas
2026-04-08 11:03:30,295 | INFO |   [121/3027] FHB    (histórico) → 502 filas

2026-04-08 11:05:10,628 | INFO |   [216/3027] DY     (histórico) → 502 filas
2026-04-08 11:05:11,660 | INFO |   [217/3027] MOD    (histórico) → 502 filas
2026-04-08 11:05:12,782 | INFO |   [218/3027] MDU    (histórico) → 502 filas
2026-04-08 11:05:13,847 | INFO |   [219/3027] MSI    (histórico) → 502 filas
2026-04-08 11:05:14,939 | INFO |   [220/3027] CTDD   (histórico) → 502 filas
2026-04-08 11:05:15,999 | INFO |   [221/3027] MYE    (histórico) → 502 filas
2026-04-08 11:05:17,053 | INFO |   [222/3027] NSSC   (histórico) → 502 filas
2026-04-08 11:05:18,084 | INFO |   [223/3027] NATH   (histórico) → 502 filas
2026-04-08 11:05:19,133 | INFO |   [224/3027] FIZZ   (histórico) → 502 filas
2026-04-08 11:05:20,232 | INFO |   [225/3027] NFG    (histórico) → 502 filas
2026-04-08 11:05:21,297 | INFO |   [226/3027] THC    (histórico) → 502 filas
2026-04-08 11:05:22,333 | INFO |   [227/3027] NRC    (histórico) → 502 filas
2026-04-08 11:05:23,383 | INFO |   [228/3027] BAC    (histórico) → 502 filas

...

2026-04-08 11:48:56,066 | INFO |   [2661/3027] VELO   (histórico) → 502 filas
2026-04-08 11:48:57,109 | INFO |   [2662/3027] ALHC   (histórico) → 502 filas
2026-04-08 11:48:58,161 | INFO |   [2663/3027] SERV   (histórico) → 502 filas
2026-04-08 11:48:59,229 | INFO |   [2664/3027] SWIM   (histórico) → 502 filas
2026-04-08 11:49:00,275 | INFO |   [2665/3027] DRS    (histórico) → 502 filas
2026-04-08 11:49:01,344 | INFO |   [2666/3027] INNV   (histórico) → 502 filas
2026-04-08 11:49:02,440 | INFO |   [2667/3027] NABL   (histórico) → 502 filas
2026-04-08 11:49:03,515 | INFO |   [2668/3027] GPAT   (histórico) → 444 filas
2026-04-08 11:49:04,579 | INFO |   [2669/3027] HAYW   (histórico) → 502 filas
2026-04-08 11:49:05,648 | INFO |   [2670/3027] PEPG   (histórico) → 502 filas
2026-04-08 11:49:06,704 | INFO |   [2671/3027] MRVL   (histórico) → 502 filas
2026-04-08 11:49:07,769 | INFO |   [2672/3027] KVYO   (histórico) → 502 filas
2026-04-08 11:49:08,811 | INFO |   [2673/3027] BETRW  (histórico

2026-04-08 11:50:52,229 | INFO |   [2767/3027] AVBP   (histórico) → 502 filas
2026-04-08 11:50:53,291 | INFO |   [2768/3027] OLPX   (histórico) → 502 filas
2026-04-08 11:50:54,341 | INFO |   [2769/3027] FLNC   (histórico) → 502 filas
2026-04-08 11:50:55,416 | INFO |   [2770/3027] BBOT   (histórico) → 502 filas
2026-04-08 11:50:56,460 | INFO |   [2771/3027] LTH    (histórico) → 502 filas
2026-04-08 11:50:57,526 | INFO |   [2772/3027] AIRS   (histórico) → 502 filas
2026-04-08 11:50:58,596 | INFO |   [2773/3027] BBUC   (histórico) → 502 filas
2026-04-08 11:50:59,645 | INFO |   [2774/3027] PTLO   (histórico) → 502 filas
2026-04-08 11:51:00,671 | INFO |   [2775/3027] TVAI   (histórico) → 192 filas
2026-04-08 11:51:01,951 | INFO |   [2776/3027] EMBC   (histórico) → 502 filas
2026-04-08 11:51:02,995 | INFO |   [2777/3027] KLC    (histórico) → 374 filas
2026-04-08 11:51:04,024 | INFO |   [2778/3027] IMMX   (histórico) → 502 filas
2026-04-08 11:51:05,621 | INFO |   [2779/3027] PDLB   (histórico

2026-04-08 11:52:45,786 | INFO |   [2873/3027] SEPN   (histórico) → 362 filas
2026-04-08 11:52:46,795 | INFO |   [2874/3027] GTEN   (histórico) → 218 filas
2026-04-08 11:52:47,826 | INFO |   [2875/3027] FVR    (histórico) → 379 filas
2026-04-08 11:52:48,876 | INFO |   [2876/3027] WAY    (histórico) → 459 filas
2026-04-08 11:52:49,920 | INFO |   [2877/3027] CGON   (histórico) → 502 filas
2026-04-08 11:52:50,994 | INFO |   [2878/3027] TE     (histórico) → 502 filas
2026-04-08 11:52:52,008 | INFO |   [2879/3027] NWE    (histórico) → 502 filas
2026-04-08 11:52:53,053 | INFO |   [2880/3027] KYTX   (histórico) → 502 filas
2026-04-08 11:52:54,092 | INFO |   [2881/3027] LB     (histórico) → 445 filas
2026-04-08 11:52:55,170 | INFO |   [2882/3027] GEV    (histórico) → 502 filas
2026-04-08 11:52:56,206 | INFO |   [2883/3027] BG     (histórico) → 502 filas
2026-04-08 11:52:57,241 | INFO |   [2884/3027] TBN    (histórico) → 446 filas
2026-04-08 11:52:58,262 | INFO |   [2885/3027] WBTN   (histórico

2026-04-08 11:54:36,651 | INFO |   [2979/3027] JCAP   (histórico) → 197 filas
2026-04-08 11:54:37,644 | INFO |   [2980/3027] MDLN   (histórico) → 76 filas
2026-04-08 11:54:38,647 | INFO |   [2981/3027] RAC    (histórico) → 243 filas
2026-04-08 11:54:39,663 | INFO |   [2982/3027] LOKV   (histórico) → 243 filas
2026-04-08 11:54:40,669 | INFO |   [2983/3027] CGCTW  (histórico) → 167 filas
2026-04-08 11:54:41,653 | INFO |   [2984/3027] BLUW   (histórico) → 173 filas
2026-04-08 11:54:42,641 | INFO |   [2985/3027] RAAQ   (histórico) → 200 filas
2026-04-08 11:54:43,643 | INFO |   [2986/3027] DAAQ   (histórico) → 200 filas
2026-04-08 11:54:44,635 | INFO |   [2987/3027] EGHA   (histórico) → 195 filas
2026-04-08 11:54:45,650 | INFO |   [2988/3027] LGN    (histórico) → 143 filas
2026-04-08 11:54:46,703 | INFO |   [2989/3027] LION   (histórico) → 481 filas
2026-04-08 11:54:47,718 | INFO |   [2990/3027] CHAC   (histórico) → 215 filas
2026-04-08 11:54:48,731 | INFO |   [2991/3027] NIQ    (histórico)


─────────────────────────────────────────────────────────────────
  OK: 3027 | FAIL: 0 | SKIP (ya al día): 0
  Total procesados: 3027

  Pipeline completo — 11:55:26

